In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [2]:
df = pd.read_csv("../data/interim/daily_clean.csv")
df['Date'] = pd.to_datetime(df['Date'])

In [4]:
df['r_oil'] = np.log(df['Brent futures'] / df['Brent futures'].shift(1))
df['r_sp500'] = np.log(df['S&P500'] / df['S&P500'].shift(1))
df['r_gold'] = np.log(df['Gold'] / df['Gold'].shift(1))
df['r_bond'] = np.log(df['US 10-year Rate'] / df['US 10-year Rate'].shift(1))

df['r_oil_pos'] = df['r_oil'].clip(lower=0)

In [ ]:
targets_ret = {
    "gold": "r_gold",
    "sp500": "r_sp500",
    "us10y": "r_bond"
}

In [18]:
results_ret = []

for name, y_col in targets_ret.items():

    df[f"{y_col}_lag"] = df[y_col].shift(1)

    df_tmp = df.dropna().copy()


    df_tmp = df.dropna().copy()

    # split
    split = int(len(df_tmp) * 0.8)
    train = df_tmp.iloc[:split]
    test = df_tmp.iloc[split:]

    # AR w/ petrol
    X_train = sm.add_constant(train[['r_oil_pos', f"{y_col}_lag"]])
    y_train = train[y_col]

    X_test = sm.add_constant(test[['r_oil_pos', f"{y_col}_lag"]])
    y_test = test[y_col]

    model = sm.OLS(y_train, X_train).fit()
    y_pred = model.predict(X_test)

    rmse = np.sqrt(((y_test - y_pred)**2).mean())
    mae = np.mean(np.abs(y_test - y_pred))

    ss_res = np.sum((y_test - y_pred)**2)
    ss_tot = np.sum((y_test - np.mean(y_test))**2)
    r2 = 1 - ss_res / ss_tot

    # AR w/o petrol
    X_train_base = sm.add_constant(train[[f"{y_col}_lag"]])
    X_test_base = sm.add_constant(test[[f"{y_col}_lag"]])

    model_base = sm.OLS(y_train, X_train_base).fit()
    y_pred_base = model_base.predict(X_test_base)

    rmse_base = np.sqrt(((y_test - y_pred_base)**2).mean())
    mae_base = np.mean(np.abs(y_test - y_pred_base))
    
    ss_res_base = np.sum((y_test - y_pred_base)**2)
    ss_tot_base = np.sum((y_test - np.mean(y_test))**2)
    r2_base = 1 - ss_res_base / ss_tot_base


    results_ret.append({
        "asset": name,
        "rmse_arx": rmse,
        "mae_arx": mae,
        "r2_arx": r2,
        "rmse_ar": rmse_base,
        "mae_ar": mae_base,
        "r2_ar": r2_base,
        "RMSE improvement (%)": 100 * (rmse_base - rmse) / rmse_base,
        "MAE improvement (%)": 100 * (mae_base - mae) / mae_base,
        "R² improvement (%)": 100 * (r2 - r2_base) / abs(r2_base) if r2_base != 0 else np.nan,
        })

results_ret = pd.DataFrame(results_ret)
print(results_ret)

   asset  rmse_arx   mae_arx    r2_arx   rmse_ar    mae_ar     r2_ar  \
0   gold  0.010363  0.007277 -0.006719  0.010352  0.007256 -0.004587   
1    spx  0.012102  0.007798  0.034481  0.012231  0.007838  0.013795   
2  us10y  0.033060  0.019610  0.018519  0.033287  0.019736  0.004983   

   RMSE improvement (%)  MAE improvement (%)  R² improvement (%)  
0             -0.106070            -0.278482          -46.487492  
1              1.054341             0.509338          149.955704  
2              0.682506             0.636933          271.636345  


In [13]:
df['vol_sp500'] = df['r_sp500']**2
df['vol_oil'] = df['r_oil_pos']**2
df['vol_gold'] = df['r_gold']**2
df['vol_bond'] = df['r_bond']**2

df['vol_oil_lag'] = df['vol_oil'].shift(1)

In [14]:
targets_vol = {
    "gold": "vol_gold",
    "sp500": "vol_sp500",
    "us10y": "vol_bond"
}

In [19]:
results_vol = []

for name, y_col in targets_vol.items():

    df[f"{y_col}_lag"] = df[y_col].shift(1)

    df_tmp = df.dropna().copy()

    # split
    split = int(len(df_tmp) * 0.8)
    train = df_tmp.iloc[:split]
    test = df_tmp.iloc[split:]

    # AR w/ petrol
    X_train = sm.add_constant(train[['vol_oil', 'vol_oil_lag', f"{y_col}_lag"]])
    y_train = train[y_col]

    X_test = sm.add_constant(test[['vol_oil', 'vol_oil_lag', f"{y_col}_lag"]])
    y_test = test[y_col]

    model = sm.OLS(y_train, X_train).fit()
    y_pred = model.predict(X_test)

    rmse = np.sqrt(((y_test - y_pred)**2).mean())
    mae = np.mean(np.abs(y_test - y_pred))

    ss_res = np.sum((y_test - y_pred)**2)
    ss_tot = np.sum((y_test - np.mean(y_test))**2)
    r2 = 1 - ss_res / ss_tot

    # AR w/o petrol
    X_train_base = sm.add_constant(train[[f"{y_col}_lag"]])
    X_test_base = sm.add_constant(test[[f"{y_col}_lag"]])

    model_base = sm.OLS(y_train, X_train_base).fit()
    y_pred_base = model_base.predict(X_test_base)

    rmse_base = np.sqrt(((y_test - y_pred_base)**2).mean())
    mae_base = np.mean(np.abs(y_test - y_pred_base))
    
    ss_res_base = np.sum((y_test - y_pred_base)**2)
    ss_tot_base = np.sum((y_test - np.mean(y_test))**2)
    r2_base = 1 - ss_res_base / ss_tot_base


    results_vol.append({
        "asset": name,
        "rmse_arx": rmse,
        "mae_arx": mae,
        "r2_arx": r2,
        "rmse_ar": rmse_base,
        "mae_ar": mae_base,
        "r2_ar": r2_base,
        "RMSE improvement (%)": 100 * (rmse_base - rmse) / rmse_base,
        "MAE improvement (%)": 100 * (mae_base - mae) / mae_base,
        "R² improvement (%)": 100 * (r2 - r2_base) / abs(r2_base) if r2_base != 0 else np.nan,
        })
    

results_vol = pd.DataFrame(results_vol)
print(results_vol)

   asset  rmse_arx   mae_arx    r2_arx   rmse_ar    mae_ar     r2_ar  \
0   gold  0.000312  0.000121  0.028242  0.000311  0.000120  0.034889   
1  sp500  0.000594  0.000166  0.142361  0.000592  0.000165  0.146839   
2  us10y  0.006420  0.001060  0.145493  0.006416  0.001061  0.146452   

   RMSE improvement (%)  MAE improvement (%)  R² improvement (%)  
0             -0.343770            -0.632972          -19.051721  
1             -0.262108            -0.371425           -3.049773  
2             -0.056185             0.069542           -0.655099  
